In [18]:
import pandas as pd
import os
print(os.getcwd())

/home/users/ntu/lizh0106


In [19]:
path_re = "/home/users/ntu/lizh0106/scratch/nscc_work/Baseline_models/results/hard_labels/XGB_nol2_tileonly_seeds/tileonly"

In [20]:
import json
import re
from pathlib import Path

import pandas as pd


REPORT_ROW_RE = re.compile(
    r"^(macro avg|weighted avg)\s+([0-9]*\.?[0-9]+)\s+([0-9]*\.?[0-9]+)\s+([0-9]*\.?[0-9]+)\s+([0-9]+)\s*$"
)

FNAME_RE = re.compile(r"^metrics_(aggc|tcga)_seed(\d+)\.json$")


def parse_classification_report(report_str: str) -> dict:
    """
    Parse sklearn-like classification_report string and extract:
    - macro avg: precision/recall/f1-score
    - weighted avg: precision/recall/f1-score
    """
    out = {
        "macro_precision": None,
        "macro_recall": None,
        "macro_f1": None,
        "weighted_precision": None,
        "weighted_recall": None,
        "weighted_f1": None,
    }

    if not report_str or not isinstance(report_str, str):
        return out

    for raw_line in report_str.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        m = REPORT_ROW_RE.match(line)
        if not m:
            continue

        row_name = m.group(1)  # 'macro avg' or 'weighted avg'
        precision = float(m.group(2))
        recall = float(m.group(3))
        f1 = float(m.group(4))
        # support = int(m.group(5))  # not requested, but available

        if row_name == "macro avg":
            out["macro_precision"] = precision
            out["macro_recall"] = recall
            out["macro_f1"] = f1
        elif row_name == "weighted avg":
            out["weighted_precision"] = precision
            out["weighted_recall"] = recall
            out["weighted_f1"] = f1

    return out


def build_metrics_df(folder_path: str | Path, save_csv: bool = True, csv_name: str = "metrics_all.csv") -> pd.DataFrame:
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    rows = []
    for fp in sorted(folder.glob("metrics_*_seed*.json")):
        m = FNAME_RE.match(fp.name)
        if not m:
            # skip unexpected names
            continue

        dataset = m.group(1)
        seed = int(m.group(2))

        with fp.open("r", encoding="utf-8") as f:
            d = json.load(f)

        row = {
            "seed": seed,
            "dataset": dataset,
            "acc_valid": d.get("acc_valid", None),
            "balanced_acc_valid": d.get("balanced_acc_valid", None),
        }

        report_str = d.get("classification_report", "")
        row.update(parse_classification_report(report_str))

        rows.append(row)

    df = pd.DataFrame(rows)

    # 让列顺序固定、好看
    desired_cols = [
        "seed", "dataset",
        "acc_valid", "balanced_acc_valid",
        "macro_precision", "macro_recall", "macro_f1",
        "weighted_precision", "weighted_recall", "weighted_f1",
    ]
    for c in desired_cols:
        if c not in df.columns:
            df[c] = None
    df = df[desired_cols].sort_values(["seed", "dataset"]).reset_index(drop=True)

    # 可选：检查是不是每个seed都有两行（aggc+tcga）
    # print(df.groupby("seed")["dataset"].nunique().value_counts())

    if save_csv:
        out_path = folder / csv_name
        df.to_csv(out_path, index=False)
        print(f"Saved CSV -> {out_path}")

    return df


# ====== 用法示例 ======
folder_path = path_re
df = build_metrics_df(folder_path, save_csv=True, csv_name="metrics_seed0_29_aggc_tcga.csv")
df.head(10)

Saved CSV -> /home/users/ntu/lizh0106/scratch/nscc_work/Baseline_models/results/hard_labels/XGB_nol2_tileonly_seeds/tileonly/metrics_seed0_29_aggc_tcga.csv


,seed,dataset,acc_valid,balanced_acc_valid,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1
0,0,aggc,0.593583,0.419487,0.47,0.42,0.44,0.58,0.59,0.58
1,0,tcga,0.450794,0.430610,0.52,0.43,0.41,0.59,0.45,0.45
2,1,aggc,0.604278,0.417156,0.48,0.42,0.43,0.58,0.60,0.59
3,1,tcga,0.450794,0.435171,0.51,0.44,0.41,0.58,0.45,0.45
4,2,aggc,0.620321,0.472774,0.52,0.47,0.49,0.60,0.62,0.61
5,2,tcga,0.444444,0.427752,0.52,0.43,0.41,0.58,0.44,0.44
6,3,aggc,0.582888,0.400839,0.44,0.40,0.41,0.57,0.58,0.57
7,3,tcga,0.434921,0.423699,0.50,0.42,0.40,0.58,0.43,0.43
8,4,aggc,0.593583,0.447366,0.48,0.45,0.46,0.59,0.59,0.59
9,4,tcga,0.453968,0.435972,0.52,0.44,0.41,0.59,0.45,0.45
